import os, subprocess

data_root = '/content/drive/MyDrive/AI_LUNG_DATA'

manifest_extracted = f'{data_root}/manifest-1600709154662'
if os.path.isdir(manifest_extracted):
    print('[SKIP] manifest already extracted.')
else:
    print('Extracting manifest.zip...')
    subprocess.run(['unzip', '-q', f'{data_root}/manifest.zip', '-d', data_root])
    print('Done.')

xml_extracted = f'{data_root}/LIDC-XML-only'
if os.path.isdir(xml_extracted):
    print('[SKIP] LIDC-XML-only already extracted.')
else:
    print('Extracting LIDC-XML-only.zip...')
    subprocess.run(['unzip', '-q', f'{data_root}/LIDC-XML-only.zip', '-d', data_root])
    print('Done.')


## Step 1: GPU Check & Mount Drive

In [ ]:
import torch

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# Unzip dataset archives — TRUE skip if folder already exists (no zip scanning)
import os, subprocess

data_root = '/content/drive/MyDrive/AI_LUNG_DATA'

# --- manifest.zip ---
manifest_extracted = f'{data_root}/manifest-1600709154662'
manifest_zip       = f'{data_root}/manifest.zip'

if os.path.isdir(manifest_extracted):
    print(f'[SKIP] manifest already extracted -> {manifest_extracted}')
else:
    print('Extracting manifest.zip -> Drive (~20 min, first run only)...')
    result = subprocess.run(['unzip', '-q', manifest_zip, '-d', data_root],
                            capture_output=True, text=True)
    print('manifest.zip done.' if result.returncode == 0 else f'ERROR: {result.stderr}')

# --- LIDC-XML-only.zip ---
xml_extracted = f'{data_root}/LIDC-XML-only'
xml_zip       = f'{data_root}/LIDC-XML-only.zip'

if os.path.isdir(xml_extracted):
    print(f'[SKIP] LIDC-XML-only already extracted -> {xml_extracted}')
else:
    print('Extracting LIDC-XML-only.zip...')
    result = subprocess.run(['unzip', '-q', xml_zip, '-d', data_root],
                            capture_output=True, text=True)
    print('LIDC-XML-only.zip done.' if result.returncode == 0 else f'ERROR: {result.stderr}')


In [ ]:
# Verify dataset layout
import os

data_root = '/content/drive/MyDrive/AI_LUNG_DATA'
assert os.path.exists(data_root), f'Dataset folder not found: {data_root}'
assert os.path.exists(f'{data_root}/manifest-1600709154662/metadata.csv'), \
    'metadata.csv missing — did unzip finish?'
assert os.path.exists(f'{data_root}/LIDC-XML-only'), \
    'LIDC-XML-only folder missing — did unzip finish?'

lidcdir = f'{data_root}/manifest-1600709154662/LIDC-IDRI'
patients = [d for d in os.listdir(lidcdir) if d.startswith('LIDC-IDRI-')]
print(f'Dataset verified. Found {len(patients)} patient folders.')

## Step 2: Clone Repository & Install Dependencies

In [ ]:
import os

repo_dir = '/content/AI_LUNG'

if os.path.exists(repo_dir):
    %cd /content/AI_LUNG
    !git pull origin main
else:
    !git clone https://github.com/KailasVS666/AI_LUNG.git /content/AI_LUNG
    %cd /content/AI_LUNG

print('Repository ready.')

In [ ]:
%cd /content/AI_LUNG
# Install requirements including MONAI and OpenCV (for CLAHE)
!pip install -q -r requirements.txt
!pip install -q -e .
print('Dependencies installed.')

## Step 3: Generate Patient-wise Data Splits

Splits patients 70/15/15 into train/val/test with no patient leakage.

In [ ]:
import os, json, shutil

split_path       = '/content/AI_LUNG/outputs/splits/patient_split.json'
drive_split_path = '/content/drive/MyDrive/AI_LUNG_DATA/patient_split.json'

os.makedirs(os.path.dirname(split_path), exist_ok=True)

if os.path.exists(split_path):
    with open(split_path) as f:
        splits = json.load(f)
    print(f'Splits already on VM — train: {len(splits["train"])}, val: {len(splits["val"])}, test: {len(splits["test"])}')

elif os.path.exists(drive_split_path):
    shutil.copy(drive_split_path, split_path)
    with open(split_path) as f:
        splits = json.load(f)
    print(f'Splits restored from Drive — train: {len(splits["train"])}, val: {len(splits["val"])}, test: {len(splits["test"])}')

else:
    !python scripts/build_splits.py \
        --dataset-root /content/drive/MyDrive/AI_LUNG_DATA/manifest-1600709154662 \
        --metadata-csv /content/drive/MyDrive/AI_LUNG_DATA/manifest-1600709154662/metadata.csv \
        --out outputs/splits/patient_split.json \
        --train-ratio 0.70 \
        --val-ratio 0.15
    shutil.copy(split_path, drive_split_path)
    print('Splits generated and saved to Drive.')

In [ ]:
# [OPTIONAL] Remap Windows paths to Colab Drive paths if split was generated locally.
# Skip if you just ran build_splits.py above.

import json, re

split_path   = '/content/AI_LUNG/outputs/splits/patient_split.json'
LOCAL_PREFIX = r'[A-Za-z]:[/\\].*?manifest-1600709154662'
COLAB_PREFIX = '/content/drive/MyDrive/AI_LUNG_DATA/manifest-1600709154662'

with open(split_path) as f:
    splits = json.load(f)

def remap(entry):
    loc = entry.get('file_location', '')
    entry['file_location'] = re.sub(LOCAL_PREFIX, COLAB_PREFIX, str(loc).replace('\\', '/'))
    return entry

for key in ('train', 'val', 'test'):
    splits[key] = [remap(e) for e in splits[key]]

with open(split_path, 'w') as f:
    json.dump(splits, f, indent=2)

sample = splits['train'][0]['file_location']
print(f'Paths remapped. Sample: {sample}')

## Step 4: Train Stage 1 — 2.5D Denoiser

**Model**: EfficientNet-B5 encoder + CBAM attention 2.5D U-Net  
**Input**: 9 consecutive low-dose simulated slices (Radon + Poisson + FBP)  
**Target**: 1 clean normal-dose central slice  
**Loss**: L1 + (1 − SSIM) + Gradient  
**Metrics**: PSNR, SSIM  
**Config**: `configs/baseline_colab.yaml`  
**Estimated time**: ~3–5 hours (T4 GPU, full dataset, 30 epochs)

In [ ]:
%cd /content/AI_LUNG
!python scripts/train_denoiser_baseline.py --config configs/baseline_colab.yaml

In [ ]:
# Show Stage 1 training results
import json
import matplotlib.pyplot as plt

history_path = '/content/drive/MyDrive/AI_LUNG_DATA/outputs/denoiser_25d/history.json'
with open(history_path) as f:
    h = json.load(f)

epochs = range(1, len(h['val_psnr']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, h['train_loss'], label='Train'); axes[0].plot(epochs, h['val_loss'], label='Val')
axes[0].set_title('Loss (L1+SSIM+Grad)'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, h['val_psnr'], 'g-o')
axes[1].set_title(f'Val PSNR (best: {max(h["val_psnr"]):.2f} dB)'); axes[1].grid(True)

axes[2].plot(epochs, h['val_ssim'], 'b-o')
axes[2].set_title(f'Val SSIM (best: {max(h["val_ssim"]):.4f})'); axes[2].grid(True)

plt.tight_layout(); plt.show()
print(f'Stage 1 — Best PSNR: {max(h["val_psnr"]):.2f} dB | Best SSIM: {max(h["val_ssim"]):.4f}')

## Step 4.5: Bridge — Export Denoised Volumes (Stage 1 → Stage 2)

This critical step runs the trained Stage 1 denoiser on every series and saves  
the denoised volumes as `.npy` files to Drive.  
Stage 2 will load these instead of raw CT volumes.

**Estimated time**: ~1–2 hours (depends on dataset size)

In [ ]:
%cd /content/AI_LUNG
!python scripts/export_denoised.py --config configs/baseline_colab.yaml
print('Denoised volumes exported to Drive.')

In [ ]:
# Verify denoised volume count
import os
den_dir = '/content/drive/MyDrive/AI_LUNG_DATA/outputs/denoised_vols'
npy_files = [f for f in os.listdir(den_dir) if f.endswith('_denoised.npy')] if os.path.exists(den_dir) else []
print(f'Denoised volumes found: {len(npy_files)}')
if npy_files:
    import numpy as np
    sample = np.load(os.path.join(den_dir, npy_files[0]))
    print(f'Sample shape: {sample.shape}, dtype: {sample.dtype}, range: [{sample.min():.3f}, {sample.max():.3f}]')

## Step 5: Train Stage 2 — 3D Reconstruction

**Model**: Lightweight 3D U-Net with full 3D-CBAM (channel + spatial attention)  
**Input**: 64×64×64 patches of denoised volumes (from Step 4.5)  
**Target**: Corresponding normal-dose volume patches  
**Loss**: L1 + SSIM3D + Gradient3D + Projection Consistency  
**Metrics**: PSNR, SSIM  
**Config**: `configs/recon3d_colab.yaml`  
**Estimated time**: ~4–6 hours (T4 GPU, full dataset, 30 epochs)

In [ ]:
%cd /content/AI_LUNG
!python scripts/train_recon3d.py --config configs/recon3d_colab.yaml

In [ ]:
# Show Stage 2 training results
import json
import matplotlib.pyplot as plt

history_path = '/content/drive/MyDrive/AI_LUNG_DATA/outputs/recon3d/history.json'
with open(history_path) as f:
    h = json.load(f)

epochs = range(1, len(h['val_psnr']) + 1)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(epochs, h['train_loss'], label='Train'); axes[0].plot(epochs, h['val_loss'], label='Val')
axes[0].set_title('Total Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, h['val_psnr'], 'g-o')
axes[1].set_title(f'Val PSNR (best: {max(h["val_psnr"]):.2f} dB)'); axes[1].grid(True)

axes[2].plot(epochs, h['val_ssim'], 'b-o')
axes[2].set_title(f'Val SSIM (best: {max(h["val_ssim"]):.4f})'); axes[2].grid(True)

if 'val_loss_proj' in h:
    axes[3].plot(epochs, h['val_loss_l1'], label='L1')
    axes[3].plot(epochs, h['val_loss_ssim'], label='SSIM')
    axes[3].plot(epochs, h['val_loss_grad'], label='Grad')
    axes[3].plot(epochs, h['val_loss_proj'], label='Proj')
    axes[3].set_title('Loss Components'); axes[3].legend(); axes[3].grid(True)

plt.tight_layout(); plt.show()
print(f'Stage 2 — Best PSNR: {max(h["val_psnr"]):.2f} dB | Best SSIM: {max(h["val_ssim"]):.4f}')

## Step 6: Train Stage 3 — Nodule Detector

**Model**: 3D CNN with CBAM attention + multi-layer classifier head  
**Loss**: Dice + Cross-Entropy  
**Metrics**: ROC AUC, Sensitivity, Specificity  
**Config**: `configs/nodule_detection_colab.yaml`  
**Estimated time**: ~3–5 hours (T4 GPU, full dataset, 30 epochs)

In [ ]:
%cd /content/AI_LUNG
!python scripts/train_nodule_detector.py --config configs/nodule_detection_colab.yaml

In [ ]:
# Show Stage 3 results
import json
import matplotlib.pyplot as plt

history_path = '/content/drive/MyDrive/AI_LUNG_DATA/outputs/nodule_detection/history.json'
with open(history_path) as f:
    h = json.load(f)

epochs = range(1, len(h['train_loss']) + 1)
aucs  = [m['auc']         for m in h['val_metrics']]
sens  = [m['sensitivity'] for m in h['val_metrics']]
spec  = [m['specificity'] for m in h['val_metrics']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, h['train_loss'], 'r-o')
axes[0].set_title('Train Loss (Dice + CE)'); axes[0].grid(True)

axes[1].plot(epochs, aucs, label=f'AUC (best: {max(aucs):.4f})', marker='o')
axes[1].plot(epochs, sens, label=f'Sensitivity (best: {max(sens):.4f})', marker='s')
axes[1].plot(epochs, spec, label=f'Specificity (best: {max(spec):.4f})', marker='^')
axes[1].set_title('Validation Metrics'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout(); plt.show()
print(f'Stage 3 — Best AUC: {max(aucs):.4f} | Sensitivity: {max(sens):.4f} | Specificity: {max(spec):.4f}')

## Step 7: Final Results Summary

In [ ]:
import json

base_dir = '/content/drive/MyDrive/AI_LUNG_DATA/outputs'

with open(f'{base_dir}/denoiser_25d/history.json')    as f: h_den = json.load(f)
with open(f'{base_dir}/recon3d/history.json')          as f: h_rec = json.load(f)
with open(f'{base_dir}/nodule_detection/history.json') as f: h_det = json.load(f)

aucs = [m['auc'] for m in h_det['val_metrics']]
sens = [m['sensitivity'] for m in h_det['val_metrics']]
spec = [m['specificity'] for m in h_det['val_metrics']]

print('=' * 63)
print('             AI_LUNG — FINAL RESULTS SUMMARY')
print('=' * 63)
print(f'  Stage 1 — Denoising      | PSNR: {max(h_den["val_psnr"]):.2f} dB | SSIM: {max(h_den["val_ssim"]):.4f}')
print(f'  Stage 2 — Reconstruction | PSNR: {max(h_rec["val_psnr"]):.2f} dB | SSIM: {max(h_rec["val_ssim"]):.4f}')
print(f'  Stage 3 — Detection      | AUC:  {max(aucs):.4f}     | Sens: {max(sens):.4f} | Spec: {max(spec):.4f}')
print('=' * 63)

## Step 8: Verify Checkpoints on Drive

In [ ]:
import os

base = '/content/drive/MyDrive/AI_LUNG_DATA/outputs'
checkpoints = [
    f'{base}/denoiser_25d/denoiser_best.pt',
    f'{base}/denoiser_25d/denoiser_last.pt',
    f'{base}/recon3d/recon3d_best.pt',
    f'{base}/recon3d/recon3d_last.pt',
    f'{base}/nodule_detection/nodule_detector_best.pt',
    f'{base}/nodule_detection/nodule_detector_last.pt',
]

all_ok = True
for cp in checkpoints:
    exists  = os.path.exists(cp)
    size_mb = os.path.getsize(cp) / 1e6 if exists else 0
    status  = f'{size_mb:.1f} MB' if exists else 'MISSING'
    print(f'  {"OK" if exists else "!!"} {os.path.basename(cp):45s} {status}')
    if not exists:
        all_ok = False

print()
print('All checkpoints saved to Drive.' if all_ok else 'Some checkpoints are missing — re-run training cells.')